# Module 03 — Lecture 1: The Leaky Integrate-and-Fire Neuron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_03_lif_neurons/01_lif_theory.ipynb)

---

Before we parallelise on the GPU, we need to deeply understand the model we are simulating. This lecture covers the LIF neuron from biophysical motivation through to numerical implementation.

**Learning objectives:**
- Derive the LIF equation from the RC circuit analogy
- Understand each parameter's biological meaning
- Implement and analyse the LIF model in pure Python (CPU)
- Characterise the f-I curve (firing rate vs input current)

## 1. From Biology to Equations

A real neuron maintains an electrical potential across its membrane (~−65 mV at rest). Input from other neurons arrives as ionic currents that depolarise the membrane. When the membrane potential crosses a threshold (~−55 mV), the neuron fires an **action potential** (spike) — a stereotyped 1 ms electrical pulse — then resets.

### The RC Circuit Analogy

The membrane behaves like a resistor-capacitor (RC) circuit:

```
                  Rm (leak)
      I(t) ──┬──[/////]──┬── EL (battery)
             │           │
            Cm           │
             │           │
             └─────────── GND

KCL: I = Cm * dV/dt + (V - EL) / Rm
```

Rearranging: $\tau_m \frac{dV}{dt} = -(V - E_L) + R_m I(t)$

where the **membrane time constant** $\tau_m = R_m C_m$.

### Adding the Spiking Rule

The LIF model adds an artificial threshold-and-reset:

$$\text{if } V(t) \geq V_{th}: \quad V \leftarrow V_{reset}, \quad t_{\text{spike}} \leftarrow t$$

And optionally, an absolute **refractory period** $\tau_{ref}$ during which V is clamped.

### Parameters and Their Biological Values

| Parameter | Symbol | Typical Value | Biological Meaning |
|-----------|--------|--------------|--------------------|
| Membrane time constant | $\tau_m$ | 20 ms | RC time scale |
| Leak reversal potential | $E_L$ | −65 mV | Resting potential |
| Membrane resistance | $R_m$ | 10 MΩ | Leak conductance |
| Spike threshold | $V_{th}$ | −55 mV | Sodium channel activation |
| Reset potential | $V_{reset}$ | −70 mV | After-hyperpolarization |
| Refractory period | $\tau_{ref}$ | 2 ms | Sodium channel inactivation |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# LIF parameters
tau_m   = 20.0   # ms
E_L     = -65.0  # mV
Rm      = 10.0   # MOhm
V_th    = -55.0  # mV
V_reset = -70.0  # mV
tau_ref = 2.0    # ms
dt      = 0.1    # ms

def simulate_lif(I_ext, T_ms=500, dt=0.1):
    """Single LIF neuron CPU simulation."""
    T = int(T_ms / dt)
    V = np.full(T, E_L)
    spikes = []
    ref_count = 0

    for t in range(1, T):
        if ref_count > 0:
            V[t] = V_reset
            ref_count -= 1
        else:
            dV = dt / tau_m * (-(V[t-1] - E_L) + Rm * I_ext)
            V[t] = V[t-1] + dV
            if V[t] >= V_th:
                V[t] = V_reset
                spikes.append(t * dt)
                ref_count = int(tau_ref / dt)

    return np.arange(T) * dt, V, spikes

# Simulate for three input current levels
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, I_ext, label in zip(axes, [1.0, 1.8, 3.0], ['Sub-threshold', 'Near-threshold', 'High-freq']):
    t, V, spikes = simulate_lif(I_ext, T_ms=300)
    ax.plot(t, V, 'b', linewidth=0.8)
    ax.axhline(V_th, color='r', linestyle='--', alpha=0.7, label=f'V_th = {V_th} mV')
    ax.axhline(E_L, color='g', linestyle=':', alpha=0.5, label=f'E_L = {E_L} mV')
    for sp in spikes:
        ax.axvline(sp, color='r', alpha=0.3, linewidth=0.5)
    fr = len(spikes) / 0.3  # Hz
    ax.set_ylabel('V (mV)', fontsize=11)
    ax.set_title(f'{label}: I = {I_ext} pA → {fr:.0f} Hz', fontsize=12)
    ax.set_ylim(-80, -45)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (ms)', fontsize=12)
plt.tight_layout()
plt.savefig('lif_traces.png', dpi=150, bbox_inches='tight')
plt.show()
print("LIF traces for three current levels.")

## 2. The f-I Curve: Firing Rate vs Input Current

The f-I curve characterizes a neuron's input-output relationship. For the LIF model, the theoretical firing rate is:

$$f(I) = \left[ \tau_{ref} + \tau_m \ln\left(\frac{R_m I - E_L + E_L}{R_m I - V_{th} + E_L}\right) \right]^{-1} \cdot 1000 \text{ Hz}$$

This only applies when $R_m I > V_{th} - E_L$ (rheobase current).

In [ ]:
# f-I curve: simulated vs analytical
I_values = np.linspace(0.5, 5.0, 50)
fr_sim = []
fr_theory = []

I_rheo = (V_th - E_L) / Rm  # rheobase: minimum I to fire

for I in I_values:
    _, _, spikes = simulate_lif(I, T_ms=2000)
    fr_sim.append(len(spikes) / 2.0)  # Hz

    if Rm * I > V_th - E_L:
        tau_isi = tau_ref + tau_m * np.log((Rm*I - (E_L - E_L)) / (Rm*I - (V_th - E_L)))
        fr_theory.append(1000.0 / tau_isi)
    else:
        fr_theory.append(0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(I_values, fr_sim, 'b-o', markersize=4, label='Simulated (Euler, dt=0.1 ms)')
ax.plot(I_values, fr_theory, 'r--', linewidth=2, label='Analytical')
ax.axvline(I_rheo, color='gray', linestyle=':', alpha=0.7, label=f'Rheobase I = {I_rheo:.2f} pA')
ax.set_xlabel('Input current I (pA)', fontsize=13)
ax.set_ylabel('Firing rate (Hz)', fontsize=13)
ax.set_title('LIF f-I Curve', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fi_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Rheobase current: {I_rheo:.2f} pA")
print(f"At I=3.0 pA: ~{fr_sim[int(np.searchsorted(I_values, 3.0))]:.0f} Hz")

## 3. Inter-Spike Interval Distribution

The ISI distribution reveals whether a neuron fires regularly or irregularly. For a constant current, the LIF fires **perfectly regularly** (all ISIs equal). With noise, ISIs become variable — more biologically realistic.

In [ ]:
def simulate_lif_noisy(I_mean, sigma, T_ms=5000, dt=0.1):
    T = int(T_ms / dt)
    V = E_L
    spikes = []
    ref_count = 0
    noise = np.random.randn(T) * sigma * np.sqrt(dt)  # Wiener noise

    for t in range(T):
        if ref_count > 0:
            V = V_reset
            ref_count -= 1
        else:
            dV = dt / tau_m * (-(V - E_L) + Rm * I_mean) + noise[t]
            V += dV
            if V >= V_th:
                spikes.append(t * dt)
                V = V_reset
                ref_count = int(tau_ref / dt)
    return spikes

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for sigma, color, label in [(0, 'b', 'No noise (regular)'),
                              (1.0, 'r', 'Moderate noise'),
                              (3.0, 'g', 'High noise')]:
    spikes = simulate_lif_noisy(2.0, sigma)
    isis = np.diff(spikes) if len(spikes) > 1 else []
    if len(isis) > 5:
        cv = np.std(isis) / np.mean(isis)
        ax1.hist(isis, bins=50, alpha=0.5, color=color,
                 label=f'{label}  CV={cv:.2f}')

ax1.set_xlabel('Inter-Spike Interval (ms)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('ISI Distributions', fontsize=13)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Coefficient of variation (CV) vs noise level
sigmas = np.linspace(0, 5, 20)
cvs = []
for sigma in sigmas:
    sp = simulate_lif_noisy(2.0, sigma, T_ms=10000)
    isis = np.diff(sp) if len(sp) > 2 else [1, 1]
    cvs.append(np.std(isis) / np.mean(isis))

ax2.plot(sigmas, cvs, 'b-o', markersize=5)
ax2.axhline(1.0, color='r', linestyle='--', alpha=0.7, label='CV=1 (Poisson)')
ax2.set_xlabel('Noise amplitude σ', fontsize=12)
ax2.set_ylabel('Coefficient of Variation (CV)', fontsize=12)
ax2.set_title('ISI Variability vs Noise', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('isi_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Numerical Integration: Euler vs Exact

For the LIF model without spikes, the exact solution is:

$$V(t) = E_L + R_m I + (V_0 - E_L - R_m I) e^{-t/\tau_m}$$

The Euler method approximation error is $O(\Delta t)$. For $\Delta t = 0.1$ ms, this is accurate enough for most neuroscience applications.

In [ ]:
# Compare Euler accuracy for different timesteps
I_ext = 2.0
T_ms = 50
V0 = -65.0

# Exact solution (no spikes, subthreshold)
t_exact = np.linspace(0, T_ms, 1000)
V_ss = E_L + Rm * I_ext  # steady-state voltage
V_exact = V_ss + (V0 - V_ss) * np.exp(-t_exact / tau_m)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_exact, V_exact, 'k-', linewidth=2, label='Exact solution')

for dt_test, color in [(1.0, 'r'), (0.5, 'orange'), (0.1, 'b'), (0.05, 'g')]:
    T = int(T_ms / dt_test)
    V = np.zeros(T)
    V[0] = V0
    for t in range(1, T):
        V[t] = V[t-1] + dt_test / tau_m * (-(V[t-1] - E_L) + Rm * I_ext)
    t_arr = np.arange(T) * dt_test
    V_exact_at_t = V_ss + (V0 - V_ss) * np.exp(-t_arr / tau_m)
    max_err = np.max(np.abs(V - V_exact_at_t))
    ax.plot(t_arr, V, '--', color=color, linewidth=1.5,
            label=f'Euler dt={dt_test} ms (max err={max_err:.3f} mV)')

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('V (mV)', fontsize=12)
ax.set_title('Euler Integration Accuracy — LIF Sub-threshold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Conclusion: dt=0.1 ms gives <0.01 mV error — sufficient for neuroscience simulations.")

## Summary

| Concept | Value |
|---------|-------|
| LIF equation | $\tau_m dV/dt = -(V-E_L) + R_m I$ |
| Rheobase current | $I_{rh} = (V_{th} - E_L) / R_m$ |
| ISI at constant I | $\text{ISI} = \tau_{ref} + \tau_m \ln(...)$ |
| Euler accuracy | Error ∝ Δt; dt=0.1 ms gives <0.01 mV error |
| CV at constant I | CV = 0 (regular); increases with noise |

**Next lecture:** Parallelise this simulation across 10,000 neurons on the GPU.

---

**Next →** [02 — Parallel LIF on GPU](02_parallel_lif_gpu.ipynb) &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_03_lif_neurons/02_parallel_lif_gpu.ipynb)